In [5]:
import pandas as pd
import numpy as np
import os

In [4]:
datasets = {
    "Amazon": ("amazon_clustering_scores.csv", "amazon_herc_clustering_scores.csv"),
    "ArXiv": ("arxiv_clustering_scores.csv", "arxiv_herc_clustering_scores.csv"),
    "DBpedia": ("db_clustering_scores.csv", "db_herc_clustering_scores.csv"),
    "RCV1": ("rcv1_clustering_scores.csv", "rcv1_herc_clustering_scores.csv"),
    "WoS": ("wos_clustering_scores.csv", "wos_herc_clustering_scores.csv"),
    "Energy_D5": ("Energy_Ecosystems_and_Humans_hierarchy_t1.0_maxsub3_depth5_synonyms0_random_clustering_scores.csv", 
                  "Energy_Ecosystems_and_Humans_hierarchy_t1.0_maxsub3_depth5_synonyms0_random_herc_direct_scores.csv"),
    "Energy_D3": ("Energy_Ecosystems_and_Humans_hierarchy_t1.0_maxsub5_depth3_synonyms0_random_clustering_scores.csv", 
                  "Energy_Ecosystems_and_Humans_hierarchy_t1.0_maxsub5_depth3_synonyms0_random_herc_direct_scores.csv"),
    "Fisheries_D5": ("Offshore_energy_impacts_on_fisheries_hierarchy_t1.0_maxsub3_depth5_synonyms0_random_clustering_scores.csv", 
                      "Offshore_energy_impacts_on_fisheries_hierarchy_t1.0_maxsub3_depth5_synonyms0_random_herc_direct_scores.csv")
}

metrics = ["ARI", "AMI", "LCA-F1", "Dendrogram_Purity"]

all_ranks = []

for name, (normal_f, herc_f) in datasets.items():
    if not os.path.exists(normal_f) or not os.path.exists(herc_f):
        print(f"Skipping {name}: File not found.")
        continue
        
    df = pd.read_csv(normal_f)
    hdf = pd.read_csv(herc_f)
    
    available_metrics = [m for m in metrics if m in df.columns and m in hdf.columns]
    
    normal_agg = df.groupby(["reduction_method", "cluster_method"])[available_metrics].mean().reset_index()
    normal_agg["config"] = normal_agg["reduction_method"] + " + " + normal_agg["cluster_method"]
    
    herc_data = {"config": "HERCULES"}
    for m in available_metrics:
        herc_data[m] = hdf[m].mean()
    herc_row = pd.DataFrame([herc_data])
    
    cols_to_keep = ["config"] + available_metrics
    combined = pd.concat([normal_agg[cols_to_keep], herc_row], ignore_index=True)
    
    for m in available_metrics:
        combined[f"{m}_rank"] = combined[m].rank(ascending=False)
    
    combined["dataset"] = name
    all_ranks.append(combined)

if all_ranks:
    master_df = pd.concat(all_ranks)
    

    rank_cols = [c for c in master_df.columns if "_rank" in c]
    final_rankings = master_df.groupby("config")[rank_cols].mean()
    

    final_rankings["Global_Mean_Rank"] = final_rankings.mean(axis=1)
    print(final_rankings.sort_values("Global_Mean_Rank"))
    final_results = final_rankings.sort_values("Global_Mean_Rank")
    
    final_results.to_csv("results.csv")
else:
    print("No data processed. Check your file paths.")

#AMI, DBPedia, F1

                        ARI_rank  AMI_rank  Global_Mean_Rank
config                                                      
UMAP + Agglomerative       1.500     1.750            1.6250
PaCMAP + Agglomerative     2.750     2.125            2.4375
Raw + Agglomerative        5.000     4.000            4.5000
PHATE + Agglomerative      8.000     7.375            7.6875
PaCMAP + HDBSCAN           7.875     7.625            7.7500
UMAP + HDBSCAN             8.125     7.625            7.8750
PHATE + DC                 7.250     8.625            7.9375
PCA + DC                   7.875     8.500            8.1875
Raw + DC                   9.500     8.000            8.7500
PCA + Agglomerative        9.250     9.125            9.1875
UMAP + DC                  9.875     9.250            9.5625
PaCMAP + DC               10.625     9.875           10.2500
tSNE + Agglomerative       9.875    11.000           10.4375
tSNE + DC                 10.375    10.750           10.5625
tSNE + HDBSCAN          

The quantitative evaluation of hierarchical clustering configurations reveals a clear performance hierarchy across the benchmark and synthetic datasets. UMAP + Agglomerative emerged as the most consistent top-performing configuration, achieving a Global Mean Rank of 1.625 across all tested metrics. PaCMAP + Agglomerative followed closely as a robust alternative with a mean rank of 2.4375. Within the diffusion-based manifold category, PHATE + Diffusion Condensation (DC) proved to be the optimal pipeline, yielding a mean rank of 7.9375 and significantly outperforming PHATE when coupled with HDBSCAN, which fell to a rank of 11.1875.

A critical finding of this study is the performance gap between manifold-learning approaches and the LLM-based HERCULES framework. Despite its capabilities, HERCULES achieved a Global Mean Rank of 11.500, as its performance degraded on professional taxonomies like RCV1 and Web of Science, where it frequently returned near-zero ARI and AMI scores. These results indicate that while zero-shot LLM partitioning is effective for structured synthetic data, manifold-based geometry remains essential for recovering complex hierarchies in real-world textual corpora. Furthermore, density-based methods like HDBSCAN consistently underperformed relative to hierarchical methods, supporting the decision to treat noise labels as a null class to prevent artificial score inflation.

In [6]:
import pandas as pd
import numpy as np
import os

datasets = {
    "Amazon": ("amazon_clustering_scores.csv", "amazon_herc_clustering_scores.csv"),
    "ArXiv": ("arxiv_clustering_scores.csv", "arxiv_herc_clustering_scores.csv"),
    "DBpedia": ("db_clustering_scores.csv", "db_herc_clustering_scores.csv"),
    "RCV1": ("rcv1_clustering_scores.csv", "rcv1_herc_clustering_scores.csv"),
    "WoS": ("wos_clustering_scores.csv", "wos_herc_clustering_scores.csv"),
    "Energy_D5": ("Energy_Ecosystems_and_Humans_hierarchy_t1.0_maxsub3_depth5_synonyms0_random_clustering_scores.csv", 
                  "Energy_Ecosystems_and_Humans_hierarchy_t1.0_maxsub3_depth5_synonyms0_random_herc_direct_scores.csv"),
    "Energy_D3": ("Energy_Ecosystems_and_Humans_hierarchy_t1.0_maxsub5_depth3_synonyms0_random_clustering_scores.csv", 
                  "Energy_Ecosystems_and_Humans_hierarchy_t1.0_maxsub5_depth3_synonyms0_random_herc_direct_scores.csv"),
    "Fisheries_D5": ("Offshore_energy_impacts_on_fisheries_hierarchy_t1.0_maxsub3_depth5_synonyms0_random_clustering_scores.csv", 
                      "Offshore_energy_impacts_on_fisheries_hierarchy_t1.0_maxsub3_depth5_synonyms0_random_herc_direct_scores.csv")
}

metrics = ["ARI", "AMI", "LCA-F1", "Dendrogram_Purity", "FM"]

all_ranks = []

for name, (normal_f, herc_f) in datasets.items():
    if not os.path.exists(normal_f) or not os.path.exists(herc_f):
        print(f"Skipping {name}: Source files not found.")
        continue
        
    df = pd.read_csv(normal_f)
    hdf = pd.read_csv(herc_f)
    
    available_metrics = [m for m in metrics if m in df.columns and m in hdf.columns]
    
    if not available_metrics:
        print(f"Skipping {name}: None of the requested metrics found in CSVs.")
        continue

    normal_agg = df.groupby(["reduction_method", "cluster_method"])[available_metrics].mean().reset_index()
    normal_agg["config"] = normal_agg["reduction_method"] + " + " + normal_agg["cluster_method"]
    
    herc_data = {"config": "HERCULES"}
    for m in available_metrics:
        herc_data[m] = hdf[m].mean()
    herc_row = pd.DataFrame([herc_data])
    
    cols_to_keep = ["config"] + available_metrics
    combined = pd.concat([normal_agg[cols_to_keep], herc_row], ignore_index=True)

    for m in available_metrics:
        combined[f"{m}_rank"] = combined[m].rank(ascending=False)
    
    combined["dataset"] = name
    all_ranks.append(combined)

if all_ranks:
    master_df = pd.concat(all_ranks)
    
    rank_cols = [c for c in master_df.columns if "_rank" in c]
    final_rankings = master_df.groupby("config")[rank_cols].mean()
    
    final_rankings["Global_Mean_Rank"] = final_rankings.mean(axis=1)
    
    final_results = final_rankings.sort_values("Global_Mean_Rank")
    
    print("\n--- Final Global Rankings ---")
    print(final_results)
    
    final_results.to_csv("results.csv")
    print("\nSuccess: results.csv has been created in the current directory.")
else:
    print("No data was processed. Please verify your CSV file paths and column names.")


--- Final Global Rankings ---
                        ARI_rank  AMI_rank  FM_rank  Global_Mean_Rank
config                                                               
UMAP + Agglomerative       1.500     1.750    2.000          1.750000
PaCMAP + Agglomerative     2.750     2.125    3.125          2.666667
Raw + Agglomerative        5.000     4.000    7.000          5.333333
PHATE + Agglomerative      8.000     7.375    6.500          7.291667
UMAP + HDBSCAN             8.125     7.625    7.125          7.625000
PaCMAP + HDBSCAN           7.875     7.625    8.000          7.833333
PHATE + DC                 7.250     8.625    8.750          8.208333
PCA + DC                   7.875     8.500    8.875          8.416667
PCA + Agglomerative        9.250     9.125    8.125          8.833333
Raw + DC                   9.500     8.000   11.000          9.500000
UMAP + DC                  9.875     9.250   11.375         10.166667
tSNE + Agglomerative       9.875    11.000   10.375        